# 01 Intent classifier: option scoring on 24 models

**Question.** How well does the option-scoring intent classifier of the SDK name the intent of a
question, on the 388 labelled prompts, for each of 24 language models?

**Why.** The intent classifier is the first node of the harness. Its choice sends the question to
a pipeline. A wrong choice makes the answer wrong, whatever the pipeline does.

**Approach.** `scripts/run_classifier.py` loads the option-scoring classifier
(`intent_harness_classifiers.option_scoring`) with ten description-only intents: the nine visual
patterns of Eyes Wide Shut (Tong et al., CVPR 2024) and `other`, with their recorded definitions
(`data/intent-definitions.json`). The model writes nothing. For each prompt it gives the
log-probability of each intent key as its reply; a softmax turns them into probabilities; the
most probable key is the answer. The confidence is 1 when all the probability is on one key and
0 when it is spread evenly. The vision-language models run on the text only. The prompts are 262
MMVP-VLM statements and 126 VLMBias prompts (`data/intent-manifest.parquet`); a random choice
scores 0.100.

**Prediction.** Accuracy rises with model size. The confidence is higher on right answers than
on wrong ones, but it stays high on wrong answers.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import runs

RECORDED = runs.recorded_dir()
if RECORDED is None:
    print("INTENT_HARNESS_RECORDED_DIR is not set: the comparison with earlier results is skipped.")
else:
    print("INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.")

INTENT_HARNESS_RECORDED_DIR is set: the comparison with earlier results runs.


## 1. Accuracy and confidence, by model

Models in order of size. One result file per model in `results/`.

In [2]:
models = json.loads((ROOT / "data" / "models.json").read_text())
rows = []
for info in models:
    table = runs.read_result(f"01-classifier-{info['model']}")
    right = table[table["predicted_intent"] == table["gold_intent"]]
    wrong = table[table["predicted_intent"] != table["gold_intent"]]
    rows.append({
        "model": info["model"],
        "type": "vision" if info["kind"] == "vlm" else "text-only",
        "GB": info["size_gb"],
        "accuracy, of 388": round(len(right) / len(table), 3),
        "confidence on right answers": round(right["confidence"].mean(), 3),
        "confidence on wrong answers": round(wrong["confidence"].mean(), 3),
    })
summary = pd.DataFrame(rows)
summary

,model,type,GB,"accuracy, of 388",confidence on right answers,confidence on wrong answers
0,smollm2-360m,text-only,0.7,0.075,0.780,0.739
1,qwen2.5-0.5b,text-only,1.0,0.322,0.843,0.608
2,qwen3-0.6b,text-only,1.5,0.495,0.826,0.701
3,llava-ov-0.5b,vision,1.8,0.178,0.609,0.441
4,gemma3-1b,text-only,2.0,0.152,0.981,0.955
5,qwen2.5-1.5b,text-only,3.1,0.598,0.941,0.786
6,smollm2-1.7b,text-only,3.4,0.338,0.557,0.443
7,qwen3-1.7b,text-only,4.1,0.629,0.989,0.924
8,qwen2.5-3b,text-only,6.2,0.691,0.969,0.829
9,phi4-mini,text-only,7.7,0.575,0.954,0.908


## 2. Accuracy by intent, for the default model `qwen3-4b`

In [3]:
table = runs.read_result("01-classifier-qwen3-4b")
table["right"] = table["predicted_intent"] == table["gold_intent"]
by_intent = table.groupby("gold_intent")["right"].agg(["sum", "count"])
by_intent["accuracy"] = (by_intent["sum"] / by_intent["count"]).round(3)
by_intent.rename(columns={"sum": "right", "count": "prompts"})

,right,prompts,accuracy
gold_intent,,,
color,24,30,0.800
orientation,17,29,0.586
other,7,12,0.583
positional,27,29,0.931
presence,16,30,0.533
quantity,121,129,0.938
state,9,30,0.300
structural,9,40,0.225
text,24,29,0.828


## 3. Comparison with the earlier results, item by item

Runs only when `INTENT_HARNESS_RECORDED_DIR` is set. For each model: the number of prompts with
the same choice as the earlier result file, and the largest difference in a log score (the
earlier scores are rounded to 4 decimals).

In [4]:
if RECORDED is not None:
    rows = []
    for info in models:
        mine = runs.read_result(f"01-classifier-{info['model']}")
        earlier = pd.read_parquet(RECORDED / f"04-choice-{info['model']}-full.parquet")
        both = mine.merge(earlier[["item_id", "predicted_intent", "raw"]], on="item_id",
                          suffixes=("", "_earlier"))
        largest_gap = 0.0
        for row in both.itertuples():
            scores = json.loads(row.log_scores)
            earlier_scores = json.loads(row.raw)
            for key in scores:
                largest_gap = max(largest_gap, abs(scores[key] - earlier_scores[key]))
        rows.append({
            "model": info["model"],
            "same choice": int((both["predicted_intent"] == both["predicted_intent_earlier"]).sum()),
            "of": len(both),
            "accuracy now": round((both["predicted_intent"] == both["gold_intent"]).mean(), 3),
            "accuracy earlier": round((both["predicted_intent_earlier"] == both["gold_intent"]).mean(), 3),
            "largest log-score gap": round(largest_gap, 4),
        })
    comparison = pd.DataFrame(rows)
    print("same choice in total:", int(comparison["same choice"].sum()), "of", int(comparison["of"].sum()))
    display(comparison)

same choice in total: 9311 of 9312


,model,same choice,of,accuracy now,accuracy earlier,largest log-score gap
0,smollm2-360m,388,388,0.075,0.075,0.0000
1,qwen2.5-0.5b,388,388,0.322,0.322,0.0000
2,qwen3-0.6b,388,388,0.495,0.495,0.0000
3,llava-ov-0.5b,388,388,0.178,0.178,0.0000
4,gemma3-1b,388,388,0.152,0.152,0.0000
5,qwen2.5-1.5b,388,388,0.598,0.598,0.0000
6,smollm2-1.7b,388,388,0.338,0.338,0.0000
7,qwen3-1.7b,388,388,0.629,0.629,0.0001
8,qwen2.5-3b,388,388,0.691,0.691,0.0001
9,phi4-mini,388,388,0.575,0.575,0.0000


## Reading the result

Accuracy rises with model size, but not in a straight line. Below 1 GB the models score 0.075 to
0.322. From 6 GB up, most models score between 0.69 and 0.78. The best model is `qwen2.5-14b`
(0.781). The two largest models, `gemma3-27b` (0.729) and `internvl3-38b` (0.760), do not score
higher than `qwen2.5-14b`.

The vision-language models read the text only here. Their scores are in the same range as the
text-only models of the same size.

The confidence on right answers is higher than on wrong answers for all 24 models. But on wrong
answers it stays high: above 0.90 for 11 of the 24 models. A high confidence does not tell that
the answer is right.

For `qwen3-4b`, the default model of the harness, `quantity` prompts get 0.938 and `structural`
prompts get 0.225. The other low intents are `state` (0.300) and `presence` (0.533).

The comparison with the earlier results gives the same choice on 9,311 of 9,312 prompts. The one
difference is `mmvp_vlm:39` ("decorations on the egg") with `llama3.2-vision-11b`. Its two best
keys, `presence` and `structural`, are 0.016 apart in log score, and the order of the two changed.
That changes the accuracy of this model from 0.740 to 0.742. The largest log-score gap of this
model (0.19) is on keys with a very small probability; for the other 23 models it is 0.0001 or
less.

## Conclusion

The option-scoring classifier of the SDK reproduces the earlier sweep on 24 models (99.99% same
choice). The best accuracy is 0.781, so about one prompt in five goes to the wrong intent when all
ten intents are loaded. The confidence does not find these wrong choices. With only the two paper
intents loaded (notebook 05), the routing problem is much smaller.